In [1]:
import pandas as pd

# NASA GISS Global Surface Temperature dataset (publicly available)
url = "https://data.giss.nasa.gov/gistemp/tabledata_v4/GLB.Ts+dSST.csv"

df = pd.read_csv(url, skiprows=1)  # skip the header comment row

# Keep only Year and annual mean temperature (J-D column = Jan to Dec average)
df = df[["Year", "J-D"]].copy()
df.columns = ["year", "temperature"]

# Drop rows with missing values
df = df[df["temperature"] != "***"]
df["temperature"] = df["temperature"].astype(float)

# Save to CSV
df.to_csv("weather_data.csv", index=False)

print("Downloaded real NASA weather data!")
print(df.head(10))
print(f"\nTotal records: {len(df)}")

Downloaded real NASA weather data!
   year  temperature
0  1880        -0.17
1  1881        -0.08
2  1882        -0.11
3  1883        -0.17
4  1884        -0.28
5  1885        -0.33
6  1886        -0.31
7  1887        -0.36
8  1888        -0.18
9  1889        -0.11

Total records: 146


In [2]:
%%writefile mapper.py
#!/usr/bin/env python3
import sys

for line in sys.stdin:
    line = line.strip()
    if line.startswith("year"):  # skip header
        continue
    try:
        parts = line.split(",")
        year = parts[0].strip()
        temp = float(parts[1].strip())
        print(f"{year}\t{temp}")
    except Exception:
        continue

Writing mapper.py


In [3]:
%%writefile reducer.py
#!/usr/bin/env python3
import sys

year_temps = {}

for line in sys.stdin:
    line = line.strip()
    try:
        year, temp = line.split("\t")
        temp = float(temp)
        if year not in year_temps:
            year_temps[year] = []
        year_temps[year].append(temp)
    except Exception:
        continue

# Calculate average temperature per year
avg_temps = {}
for year, temps in year_temps.items():
    avg_temps[year] = sum(temps) / len(temps)

# Print all year averages
print("\n--- Average Temperature Per Year ---")
for year in sorted(avg_temps):
    print(f"Year: {year}  |  Avg Temp: {avg_temps[year]:.2f}°C")

# Find coolest and hottest year
coolest_year = min(avg_temps, key=avg_temps.get)
hottest_year = max(avg_temps, key=avg_temps.get)

print("\n--- RESULT ---")
print(f"Coolest Year: {coolest_year}  |  Avg Temp: {avg_temps[coolest_year]:.2f}°C")
print(f"Hottest Year: {hottest_year}  |  Avg Temp: {avg_temps[hottest_year]:.2f}°C")

Writing reducer.py


In [4]:
import subprocess

result = subprocess.run(
    "type weather_data.csv | python mapper.py | python reducer.py",
    shell=True,
    capture_output=True,
    text=True
)

print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)


--- Average Temperature Per Year ---
Year: 1880  |  Avg Temp: -0.17°C
Year: 1881  |  Avg Temp: -0.08°C
Year: 1882  |  Avg Temp: -0.11°C
Year: 1883  |  Avg Temp: -0.17°C
Year: 1884  |  Avg Temp: -0.28°C
Year: 1885  |  Avg Temp: -0.33°C
Year: 1886  |  Avg Temp: -0.31°C
Year: 1887  |  Avg Temp: -0.36°C
Year: 1888  |  Avg Temp: -0.18°C
Year: 1889  |  Avg Temp: -0.11°C
Year: 1890  |  Avg Temp: -0.36°C
Year: 1891  |  Avg Temp: -0.22°C
Year: 1892  |  Avg Temp: -0.27°C
Year: 1893  |  Avg Temp: -0.31°C
Year: 1894  |  Avg Temp: -0.31°C
Year: 1895  |  Avg Temp: -0.23°C
Year: 1896  |  Avg Temp: -0.12°C
Year: 1897  |  Avg Temp: -0.11°C
Year: 1898  |  Avg Temp: -0.28°C
Year: 1899  |  Avg Temp: -0.18°C
Year: 1900  |  Avg Temp: -0.09°C
Year: 1901  |  Avg Temp: -0.16°C
Year: 1902  |  Avg Temp: -0.28°C
Year: 1903  |  Avg Temp: -0.37°C
Year: 1904  |  Avg Temp: -0.48°C
Year: 1905  |  Avg Temp: -0.26°C
Year: 1906  |  Avg Temp: -0.22°C
Year: 1907  |  Avg Temp: -0.39°C
Year: 1908  |  Avg Temp: -0.43°C
Year: